In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, cohen_kappa_score, recall_score
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 1. CONFIGURACIÓN ---
ruta_entrenamiento = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion reducida\train_final.csv"
ruta_prueba = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion reducida\test_final.csv"
ruta_externa = r"C:\Users\benja\Desktop\BD PAMPA\Calculos fusion reducida\external_final.csv"
nombre_archivo_reporte = "REPORTE_FINAL_(SIN_LOGPcons).txt"

# --- 2. LÓGICA DE EJECUCIÓN ---
try:
    print("Cargando datasets finales...")
    df_train = pd.read_csv(ruta_entrenamiento, sep=',')
    df_test = pd.read_csv(ruta_prueba, sep=',')
    df_external = pd.read_csv(ruta_externa, sep=',')
    
    atributos_seleccionados = df_train.columns[:-1].tolist()
    columna_clase = df_train.columns[-1]
    
    X_train, y_train = df_train[atributos_seleccionados], df_train[columna_clase]
    X_test, y_test = df_test[atributos_seleccionados], df_test[columna_clase]
    X_external, y_external = df_external[atributos_seleccionados], df_external[columna_clase]
    print(f"Datasets cargados con {len(atributos_seleccionados)} atributos.")

    # --- Optimización ---
    param_grid = {'n_estimators': [100, 200, 300], 'max_features': ['sqrt', 'log2'],
                  'max_depth': [10, 20, None], 'min_samples_split': [2, 5]}
    
    print("\nIniciando optimización del modelo (GridSearchCV)...")
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    grid_search = GridSearchCV(RandomForestClassifier(random_state=42), 
                               param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train)
    
    modelo_final = grid_search.best_estimator_
    print("Optimización completada.")

    # --- Validación Final y Reporte ---
    print("\nEvaluando y generando reporte final...")
    with open(nombre_archivo_reporte, 'w') as f:
        f.write("--- REPORTE FINAL DEL PROYECTO (SIN LOGPcons) ---\n")
        f.write(f"Modelo: Random Forest\n")
        f.write(f"Atributos: {len(atributos_seleccionados)}\n")
        f.write("\n--- Hiperparámetros Óptimos Encontrados ---\n")
        f.write(str(grid_search.best_params_) + "\n")
        
        for nombre_set, X_eval, y_eval in [("Prueba Interna (Test Set)", X_test, y_test), 
                                           ("Prueba Externa (External Set)", X_external, y_external)]:
            y_pred, y_proba = modelo_final.predict(X_eval), modelo_final.predict_proba(X_eval)[:, 1]
            f.write(f"\n--- Resultados en: {nombre_set} ---\n")
            f.write("-" * 40 + "\n")
            f.write(f"  Accuracy:         {accuracy_score(y_eval, y_pred):.4f}\n")
            f.write(f"  ROC AUC:          {roc_auc_score(y_eval, y_proba):.4f}\n")
            f.write(f"  BACC:             {balanced_accuracy_score(y_eval, y_pred):.4f}\n")
            f.write(f"  Sensitivity:      {recall_score(y_eval, y_pred, pos_label='Act1'):.4f}\n")
            f.write(f"  Specificity:      {recall_score(y_eval, y_pred, pos_label='Act-1'):.4f}\n")
            f.write(f"  Kappa:            {cohen_kappa_score(y_eval, y_pred):.4f}\n")
    
    print(f"\n✅ ¡PROYECTO COMPLETADO! Revisa el archivo '{nombre_archivo_reporte}'.")

except Exception as e:
    print(f"\n❌ Ocurrió un error inesperado: {e}")

Cargando datasets finales...
Datasets cargados con 23 atributos.

Iniciando optimización del modelo (GridSearchCV)...
Fitting 10 folds for each of 36 candidates, totalling 360 fits
Optimización completada.

Evaluando y generando reporte final...

✅ ¡PROYECTO COMPLETADO! Revisa el archivo 'REPORTE_FINAL_(SIN_LOGPcons).txt'.
